# Sub-session S015-S019: Preprocessing & Baseline Model
*Note: We are using XGBoost and pure Pandas to bypass a bug in the latest version of scikit-learn on Python 3.14!*

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

print("Libraries successfully imported!")

### 1. Encode Target Labels
Machine Learning needs numbers, not words. We map 'apple' -> 0, 'banana' -> 1, etc.

In [ ]:
df = pd.read_csv('../data/raw/crop_recommendation.csv')

crops = df['label'].unique()
crop_map = {crop: i for i, crop in enumerate(crops)}
df['label_encoded'] = df['label'].map(crop_map)

print(f"Encoded {len(crops)} unique crops to numbers.")

### 2. The Train/Test Split
We hide 20% of the data to test the model later.

In [ ]:
# Take a random 80% of rows for training
train_df = df.sample(frac=0.8, random_state=42)
# The remaining 20% goes to testing
test_df = df.drop(train_df.index)

features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

X_train = train_df[features]
y_train = train_df['label_encoded']

X_test = test_df[features]
y_test = test_df['label_encoded']

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

### 3. Preprocessing (Standard Scaling)
We calculate the mean and standard deviation ONLY from the training data, and apply that same scale to the test data. This avoids 'Data Leakage'.

In [ ]:
train_mean = X_train.mean()
train_std = X_train.std()

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

print("Data scaled successfully!")

### 4. Train the Baseline Model (XGBoost)
XGBoost is an extremely powerful algorithm that trains hundreds of decision trees sequentially.

In [ ]:
# Initialize and train the model
model = xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train_scaled, y_train)

# Predict on the test data
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = (y_pred == y_test).mean()
print(f"🎉 XGBoost Model Accuracy: {accuracy * 100:.2f}%")